# ***Projeto Oraculum***: Análise de Viabilidade de Terreno

Este notebook contém o fluxo de análise do projeto Oraculum. O objetivo é processar imagens de satélite e de relevo para gerar um score de viabilidade para construção civil.

## 1. Configuração do Ambiente

 1. Abra o Terminal (ALT + F12 no PyCharm) e execute '***pip install -r requirements.txt***'
 2. Inicie a autenticação com o projeto no Google Cloud

> (Caso o terminal dê errado, execute a célula abaixo)

In [ ]:
'''Caso dê errado ou preguiça:'''
#Instalação de dependências
!pip install numpy opencv-python matplotlib tifffile imagecodecs rasterio earthengine-api geemap ipywidgets

import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import time
from datetime import datetime
import re
import ipywidgets
from ipywidgets import interact, IntSlider, FloatSlider, VBox

# Biblioteca principal para interagir com o Google Earth Engine
import ee
import geemap

# Bibliotecas especializadas para arquivos GeoTIFF
import tifffile
import rasterio
from rasterio.warp import reproject, Resampling

#Nossas próprias bibliotecas
import processamento
import utilitarios
import obter_dados

In [ ]:
ID_PROJETO_GOOGLE = 'oraculum-eseg'
obter_dados.autenticar_ee(ID_PROJETO_GOOGLE)

## 2. Definição da Área e Aquisição de Dados

Aqui definimos o ponto central e o tamanho da área que desejamos analisar. As funções do módulo `utilitarios` convertem as coordenadas para o formato correto e criam um polígono da área de interesse (AOI). Em seguida, o módulo `obter_dados` utiliza essa AOI para baixar os dados de satélite e relevo do Google Earth Engine.

In [ ]:
# --- 1. DEFINA O LOCAL E TAMANHO ---
lat_dms = "20°14'58.2\"S"
lon_dms = "46°59'53.7\"W"
tamanho_km = 10

# --- 2. CONVERTER E CRIAR A ÁREA ---
lat_dd = utilitarios.dms_para_dd(lat_dms)
lon_dd = utilitarios.dms_para_dd(lon_dms)
coords_poligono = utilitarios.criar_bounding_box(lat_dd, lon_dd, tamanho_km)
area_de_interesse = ee.Geometry.Polygon(coords_poligono)

# --- 3. DEFINIR PASTA DE SAÍDA E BAIXAR OS DADOS ---
# A função agora retorna o caminho da pasta onde os arquivos foram salvos.
caminho_arquivo_satelite, caminho_arquivo_relevo = obter_dados.baixar_dados_da_area(
    area_de_interesse,
    pasta_mae="outputs"
)

print(f"\nArquivos para esta análise:")
print(f"Satélite: {caminho_arquivo_satelite}")
print(f"Relevo: {caminho_arquivo_relevo}")

In [ ]:
# --- CONFIGURAÇÃO DA ANÁLISE ---
# Para analisar um conjunto de dados, cole o timestamp da pasta aqui.
# Se você acabou de baixar, use o timestamp que apareceu na saída da célula anterior.
# Se quer analisar dados antigos, copie o nome da pasta de 'outputs'.

timestamp_da_analise = "20251013_163542" # ⇽ ÚNICO LUGAR PARA MUDAR

# --- Construção automática dos caminhos ---
pasta_da_sessao = os.path.join("outputs", timestamp_da_analise)
caminho_arquivo_satelite = os.path.join(pasta_da_sessao, f"{timestamp_da_analise}_satelite.tif")
caminho_arquivo_relevo = os.path.join(pasta_da_sessao, f"{timestamp_da_analise}_relevo.tif")

print(f"Pronto para analisar a sessão: {timestamp_da_analise}")
print(f"Verificando se os arquivos existem...")

# Checagem de segurança
if os.path.exists(caminho_arquivo_satelite) and os.path.exists(caminho_arquivo_relevo):
    print(">>> SUCESSO! Arquivos encontrados. Pode prosseguir com as próximas células de análise.")
else:
    print(">>> ERRO! Um ou ambos os arquivos não foram encontrados. Verifique o timestamp ou se você baixou o arquivo do Google Drive.")

### 3. Carga e Alinhamento de Dados

In [ ]:
# --- Célula de Carga e Alinhamento (Versão Final Sincronizada) ---

# A função agora retorna 5 produtos, que armazenamos em suas respectivas variáveis
(
    IMG_SATELITE,
    IMG_RELEVO,
    BANDA_VERDE_BRUTA,
    BANDA_NIR_BRUTA,
    nodata_relevo
) = processamento.alinhar_imagens(
    caminho_arquivo_satelite,
    caminho_arquivo_relevo
)

print("\nDados carregados e prontos para análise:")
print(f"- Imagem Visual (RGB): {IMG_SATELITE.shape}")
print(f"- Mapa de Relevo: {IMG_RELEVO.shape}")
print(f"- Banda Verde (Bruta): {BANDA_VERDE_BRUTA.shape}")
print(f"- Banda NIR (Bruta): {BANDA_NIR_BRUTA.shape}")


# --- VISUALIZAÇÃO DE CONFIRMAÇÃO ---
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Dados Finais Prontos para Análise', fontsize=16)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title(f'Satélite Alinhado (RGB)')
ax[0].axis('off')

# Lógica para visualização correta do relevo
dados_validos_relevo = IMG_RELEVO != nodata_relevo
min_relevo = IMG_RELEVO[dados_validos_relevo].min()
max_relevo = IMG_RELEVO[dados_validos_relevo].max()
ax[1].imshow(IMG_RELEVO, cmap='viridis', vmin=min_relevo, vmax=max_relevo)
ax[1].set_title(f'Relevo Alinhado (Visual)')
ax[1].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# --- Célula de Teste e Salvamento da Visualização V1 (Obsoleta) ---

print("Gerando e salvando a visualização 3D (v1)...")

# Chama a função v1 (obsoleta) que mantivemos para registro histórico
vis_v1 = processamento.fundir_imagens_v1(IMG_SATELITE, IMG_RELEVO)

# --- Exibir e Salvar ---
plt.figure(figsize=(12, 12))
plt.imshow(vis_v1)
plt.title('Visualização 3D - V1 (Método Simples)')
plt.axis('off')

# Constrói o caminho de saída DENTRO da pasta da sessão
# A variável 'pasta_da_sessao' foi criada na célula de aquisição de dados
nome_arquivo_saida = os.path.join(pasta_da_sessao, 'visualizacao_3D_v1.png')

# Salva a figura em alta qualidade
plt.savefig(nome_arquivo_saida, dpi=300, bbox_inches='tight', pad_inches=0.1)

print(f"Imagem da visualização V1 salva em: {nome_arquivo_saida}")

# Mostra a figura no notebook
plt.show()

In [ ]:
# --- Célula de Teste e Salvamento da Visualização V2 (Hillshade) ---

print("--- Testando e salvando a função de visualização 3D definitiva (Hillshade) ---")

# Chama a nova função final que usa Hillshade
imagem_3d_final = processamento.fundir_imagens_v2_3D(IMG_SATELITE, IMG_RELEVO)

# --- Exibir e Salvar ---
plt.figure(figsize=(12, 12))
plt.imshow(imagem_3d_final)
plt.title('Visualização 3D Final (Hillshade + HSV)')
plt.axis('off')

# Constrói o caminho de saída DENTRO da pasta da sessão
# A variável 'pasta_da_sessao' foi criada na célula de aquisição de dados
nome_arquivo_saida = os.path.join(pasta_da_sessao, 'visualizacao_3D_v2.png')

# Salva a figura em alta qualidade
plt.savefig(nome_arquivo_saida, dpi=300, bbox_inches='tight', pad_inches=0.1)

print(f"Imagem da visualização V2 (Hillshade + HSV) salva em: {nome_arquivo_saida}")

# Mostra a figura no notebook
plt.show()

## 4. Análise de Hidrografia

In [ ]:
# --- ETAPA 2.1: Análise de Hidrografia (Comparativo V1 vs V2) ---

# Chama ambas as funções de processamento para gerar os mapas
MAPA_AGUA_V1 = processamento.criar_mapa_agua_v1(IMG_SATELITE)
MAPA_AGUA_V2 = processamento.criar_mapa_agua_v2(IMG_SATELITE)

# Define qual versão será a "oficial" para as próximas etapas do projeto
MAPA_AGUA = MAPA_AGUA_V2
print("Mapas de água V1 e V2 criados. A versão V2 foi definida como a oficial (MAPA_AGUA).")

# --- VISUALIZAÇÃO COMPARATIVA ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Comparativo dos Métodos de Detecção de Água', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL')
ax[0].axis('off')

ax[1].imshow(MAPA_AGUA_V1, cmap='gray')
ax[1].set_title('V1 - Método HSV')
ax[1].axis('off')

ax[2].imshow(MAPA_AGUA_V2, cmap='gray')
ax[2].set_title('V2 - Método RGB')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- SALVANDO OS RESULTADOS ---
caminho_v1 = os.path.join(pasta_da_sessao, 'mapa_agua_v1.png')
caminho_v2 = os.path.join(pasta_da_sessao, 'mapa_agua_v2.png')
plt.imsave(caminho_v1, MAPA_AGUA_V1, cmap='gray')
plt.imsave(caminho_v2, MAPA_AGUA_V2, cmap='gray')
print(f"Resultados da análise de hidrografia salvos na pasta da sessão.")

In [ ]:
def visualizar_mapa_agua_ajustavel(max_intensidade, min_dif_azul, min_dif_verde):
    # Chama a nossa nova função v3 com os valores dos sliders
    mapa_gerado = processamento.criar_mapa_agua_v3_ajustavel(
        IMG_SATELITE,
        limiar_intensidade=max_intensidade,
        diferenca_azul=min_dif_azul,
        diferenca_verde=min_dif_verde
    )

    # Plota o resultado
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(IMG_SATELITE)
    ax[0].set_title('Original')
    ax[0].axis('off')

    ax[1].imshow(mapa_gerado, cmap='gray')
    ax[1].set_title('Mapa de Água Resultante')
    ax[1].axis('off')

    plt.show()

# Cria o widget interativo que chama a função de visualização
interact(
    visualizar_mapa_agua_ajustavel,
    max_intensidade=IntSlider(value=100, min=50, max=150, step=5, description='Max Intensidade:'),
    min_dif_azul=IntSlider(value=15, min=0, max=50, step=1, description='Min Dif. Azul-Verm:'),
    min_dif_verde=IntSlider(value=5, min=0, max=50, step=1, description='Min Dif. Verde-Verm:')
);

In [ ]:
print("Análise de Hidrografia (NDWI com dados brutos)...")

# 1. Usa as bandas BRUTAS (BANDA_VERDE_BRUTA e BANDA_NIR_BRUTA) para o cálculo
banda_verde = BANDA_VERDE_BRUTA.astype(float)
banda_nir = BANDA_NIR_BRUTA.astype(float)

# 2. Calcula o NDWI
numerador = banda_verde - banda_nir
denominador = banda_verde + banda_nir
ndwi = np.divide(numerador, denominador, out=np.zeros_like(numerador), where=denominador!=0)

# 3. Cria a máscara de água com um limiar
limiar = 0.1
mascara_ndwi = ndwi > limiar
MAPA_AGUA = mascara_ndwi.astype(np.uint8) * 255
print("Mapa de Hidrografia (NDWI) criado com sucesso!")

# --- VISUALIZAÇÃO COMPLETA DO PROCESSO ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Processo de Análise de Hidrografia via NDWI', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL (RGB)')
ax[0].axis('off')

im = ax[1].imshow(ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
ax[1].set_title('MAPA DE NDWI (CIENTÍFICO)')
ax[1].axis('off')
fig.colorbar(im, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04)

ax[2].imshow(MAPA_AGUA, cmap='gray')
ax[2].set_title('MAPA DE ÁGUA FINAL (FEATURE 1)')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# --- SALVANDO O RESULTADO ---
caminho_saida_hidro = os.path.join(pasta_da_sessao, 'mapa_agua_ndwi.png')
plt.imsave(caminho_saida_hidro, MAPA_AGUA, cmap='gray')
print(f"\nMapa de hidrografia (NDWI) salvo em: {caminho_saida_hidro}")

plt.show()

In [ ]:
# Esta função interna será chamada toda vez que você mexer no slider
def visualizar_limiar_ndwi(limiar):
    mascara_ndwi = ndwi > limiar
    mapa_agua_gerado = mascara_ndwi.astype(np.uint8) * 255

    fig, ax = plt.subplots(1, 2, figsize=(16, 8))

    # --- MUDANÇA AQUI ---
    # Trocamos o mapa de cores para 'Blues', que é mais intuitivo para água.
    # Ajustamos vmin e vmax para focar nos valores positivos, onde a água está.
    im = ax[0].imshow(ndwi, cmap='Blues', vmin=-0.5, vmax=1)
    ax[0].set_title('MAPA DE NDWI (Escala de Azuis)')
    ax[0].axis('off')

    ax[1].imshow(mapa_agua_gerado, cmap='gray')
    ax[1].set_title(f'Resultado para Limiar = {limiar:.2f}')
    ax[1].axis('off')

    plt.show()

# Cria o widget interativo com um slider para o limiar
interact(
    visualizar_limiar_ndwi,
    limiar=FloatSlider(value=0.1, min=-0.2, max=0.4, step=0.01, description='Limiar NDWI:')
);

In [ ]:
# --- ETAPA 2.1: Análise de Hidrografia (Versão Modular) ---

# Chama a nossa nova função de processamento, usando as bandas brutas que já temos.
# Ela retorna tanto a máscara final quanto o mapa de NDWI para visualização.
MAPA_AGUA, ndwi_mapa = processamento.criar_mapa_agua_v4_ndwi(BANDA_VERDE_BRUTA, BANDA_NIR_BRUTA, limiar=0.0)


# --- VISUALIZAÇÃO COMPLETA DO PROCESSO ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Processo de Análise de Hidrografia via NDWI', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL (RGB)')
ax[0].axis('off')

# Mostra o mapa de NDWI que a função retornou
im = ax[1].imshow(ndwi_mapa, cmap='RdYlBu', vmin=-1, vmax=1)
ax[1].set_title('MAPA DE NDWI (CIENTÍFICO)')
ax[1].axis('off')
fig.colorbar(im, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04)

# Mostra a máscara final que a função retornou
ax[2].imshow(MAPA_AGUA, cmap='gray')
ax[2].set_title('MAPA DE ÁGUA FINAL (FEATURE 1)')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# --- SALVANDO OS RESULTADOS ---
caminho_saida_mascara = os.path.join(pasta_da_sessao, 'mapa_agua_ndwi_calibrado.png')
caminho_saida_comparacao = os.path.join(pasta_da_sessao, 'comparacao_hidrografia.png')

plt.imsave(caminho_saida_mascara, MAPA_AGUA, cmap='gray')
fig.savefig(caminho_saida_comparacao, dpi=300, bbox_inches='tight')
print(f"\nResultados da análise de hidrografia salvos na pasta da sessão.")

plt.show()

## Área de Teste